# Datathon 7MLET - Grupo 13
## Plataforma de Experimentacao Adaptativa para Ofertas com Multi-Armed Bandit

**Aluna:** Giovanna Catelli

**Dataset:** [bank-marketing (henriqueyamahata)](https://www.kaggle.com/datasets/henriqueyamahata/bank-marketing)

---

## Secao 0: Setup e Dependencias

In [ ]:
# Instalar dependencias que nao vem no Kaggle
!pip install mlflow -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import mlflow
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print('Setup concluido com sucesso!')

---
## Secao 1: Carregamento dos Dados

In [ ]:
import os

# Tentar path do Kaggle primeiro, depois path local
possible_paths = [
    '/kaggle/input/bank-marketing/bank-additional-full.csv',
    '/kaggle/input/bank-marketing/bank.csv',
    '/kaggle/input/bank-marketing/bank-full.csv',
    '../data/bank-additional-full.csv',
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path, sep=';')
        print(f'Dados carregados de: {path}')
        break

if df is None:
    raise FileNotFoundError('Dataset nao encontrado. Adicione bank-marketing como Input no Kaggle.')

print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
df.head()

---
## Secao 2: EDA (Analise Exploratoria de Dados)

In [ ]:
# Informacoes gerais do dataset
print('=' * 60)
print('INFORMACOES DO DATASET')
print('=' * 60)
print(f'\nShape: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'\nTipos de dados:')
print(df.dtypes)
print(f'\nValores nulos:')
print(df.isnull().sum())
print(f'\nEstatisticas descritivas:')
df.describe()

In [ ]:
# Analise do target (variavel y)
print('=' * 60)
print('DISTRIBUICAO DO TARGET')
print('=' * 60)

target_col = 'y' if 'y' in df.columns else 'deposit'
print(f'\nColuna target: {target_col}')
print(f'\nDistribuicao:')
print(df[target_col].value_counts())
print(f'\nProporcao:')
print(df[target_col].value_counts(normalize=True).round(4))

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
df[target_col].value_counts().plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('Distribuicao do Target (Conversao)')
ax.set_xlabel('Subscreveu deposito a prazo?')
ax.set_ylabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao de variaveis numericas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

n_rows = len(num_cols) // 3 + 1
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df[col].hist(bins=30, ax=axes[i], color='#3498db', edgecolor='white')
    axes[i].set_title(f'Distribuicao: {col}')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao de variaveis categoricas
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
if target_col in cat_cols:
    cat_cols.remove(target_col)

n_rows = len(cat_cols) // 2 + 1
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    df[col].value_counts().head(10).plot(kind='barh', ax=axes[i], color='#9b59b6')
    axes[i].set_title(f'{col}')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Taxa de conversao por variaveis categoricas principais
df['converted'] = (df[target_col] == 'yes').astype(int)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

key_cats = ['job', 'marital', 'education', 'contact']
for ax, col in zip(axes.flatten(), key_cats):
    if col in df.columns:
        conv_rate = df.groupby(col)['converted'].mean().sort_values(ascending=False)
        conv_rate.plot(kind='bar', ax=ax, color='#e67e22')
        ax.set_title(f'Taxa de Conversao por {col}')
        ax.set_ylabel('Taxa')
        ax.axhline(y=df['converted'].mean(), color='red', linestyle='--', label='Media geral')
        ax.legend()

plt.tight_layout()
plt.show()

---
## Secao 3: Preparacao de Dados e Feature Engineering

In [ ]:
# Remover coluna 'duration' (vazamento temporal)
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])
    print('Coluna duration removida (vazamento temporal)')

print(f'Shape apos remocao: {df.shape}')

In [ ]:
# Tratar valores 'unknown'
print('Valores unknown por coluna:')
for col in df.columns:
    if df[col].dtype == 'object':
        unknowns = (df[col] == 'unknown').sum()
        if unknowns > 0:
            print(f'  {col}: {unknowns} ({unknowns/len(df)*100:.1f}%)')

In [ ]:
# Feature Engineering
# Criar faixa etaria
df['faixa_etaria'] = pd.cut(df['age'], bins=[0, 30, 45, 55, 100],
                            labels=['jovem', 'adulto', 'senior', 'idoso'])

# Flag de contato anterior
if 'pdays' in df.columns:
    df['teve_contato_anterior'] = (df['pdays'] != -1).astype(int) if -1 in df['pdays'].values else (df['pdays'] != 999).astype(int)

# Flag de sucesso anterior
if 'poutcome' in df.columns:
    df['sucesso_anterior'] = (df['poutcome'] == 'success').astype(int)

print('Features criadas:')
print('  - faixa_etaria')
print('  - teve_contato_anterior')
print('  - sucesso_anterior')
df[['age', 'faixa_etaria', 'teve_contato_anterior', 'sucesso_anterior']].head(10)

In [ ]:
# Encoding de variaveis categoricas
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
label_encoders = {}

cat_cols_encode = df_encoded.select_dtypes(include=['object', 'category']).columns.tolist()
if target_col in cat_cols_encode:
    cat_cols_encode.remove(target_col)

for col in cat_cols_encode:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

# Target binario
df_encoded['target'] = (df_encoded[target_col] == 'yes').astype(int)

print(f'Dataset codificado: {df_encoded.shape}')
df_encoded.head()

In [ ]:
# Definicao dos bracos (ofertas) do Bandit
# Simular 4 bracos representando diferentes estrategias de oferta

BRACOS = {
    0: 'Deposito a prazo (padrao)',
    1: 'Deposito com taxa premium',
    2: 'Produto alternativo (emprestimo)',
    3: 'Controle (nao contatar)'
}

print('Bracos definidos:')
for k, v in BRACOS.items():
    print(f'  Braco {k}: {v}')

# Definir funcao de reward simulado baseada no perfil do cliente e braco
def simular_reward(row, braco):
    """
    Simula a recompensa (conversao) dado um cliente e um braco escolhido.
    Usa a variavel target real + ajustes por contexto.
    """
    base_prob = row['target']  # 1 se converteu, 0 se nao
    
    if braco == 0:  # Deposito padrao
        return base_prob
    elif braco == 1:  # Deposito premium - melhor para alto saldo
        if 'balance' in row.index and row.get('balance', 0) > 1500:
            return min(1.0, base_prob * 1.3)
        return base_prob * 0.7
    elif braco == 2:  # Emprestimo - melhor para jovens
        if 'age' in row.index and row.get('age', 40) < 35:
            return min(1.0, base_prob * 1.2)
        return base_prob * 0.5
    elif braco == 3:  # Controle
        return 0.0
    return 0.0

print('\nFuncao de reward definida.')

In [ ]:
# Split dos dados: treino/simulacao (70%) e teste/avaliacao (30%)
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df_encoded, test_size=0.3, random_state=42, stratify=df_encoded['target'])

print(f'Dados de treino/simulacao: {df_train.shape[0]} registros')
print(f'Dados de teste/avaliacao: {df_test.shape[0]} registros')
print(f'\nTaxa de conversao treino: {df_train["target"].mean():.4f}')
print(f'Taxa de conversao teste: {df_test["target"].mean():.4f}')

---
## Secao 4: Baseline (Regra Fixa)

O baseline e uma politica deterministica que sempre recomenda o mesmo braco (aquele com maior taxa de conversao historica). Serve como referencia para comparar com o algoritmo adaptativo.

In [ ]:
# Calcular taxa de conversao por braco no conjunto de treino
print('=' * 60)
print('BASELINE: REGRA FIXA (MELHOR BRACO HISTORICO)')
print('=' * 60)

# Simular rewards para cada braco usando dados de treino
taxas_por_braco = {}
for braco in BRACOS.keys():
    rewards = df_train.apply(lambda row: simular_reward(row, braco), axis=1)
    taxas_por_braco[braco] = rewards.mean()
    print(f'  Braco {braco} ({BRACOS[braco]}): taxa = {rewards.mean():.4f}')

melhor_braco_fixo = max(taxas_por_braco, key=taxas_por_braco.get)
print(f'\nMelhor braco fixo: {melhor_braco_fixo} ({BRACOS[melhor_braco_fixo]})')
print(f'Taxa de conversao do baseline: {taxas_por_braco[melhor_braco_fixo]:.4f}')

In [ ]:
# Simular o baseline no conjunto de teste
baseline_rewards = []
baseline_regrets = []

for idx, row in df_test.iterrows():
    # Baseline sempre escolhe o melhor braco fixo
    reward = simular_reward(row, melhor_braco_fixo)
    baseline_rewards.append(reward)
    
    # Regret: melhor reward possivel - reward obtido
    best_reward = max(simular_reward(row, b) for b in BRACOS.keys())
    baseline_regrets.append(best_reward - reward)

baseline_ctr = np.mean(baseline_rewards)
baseline_total_reward = np.sum(baseline_rewards)
baseline_total_regret = np.sum(baseline_regrets)

print(f'\nResultados do Baseline no teste:')
print(f'  CTR (taxa de conversao): {baseline_ctr:.4f}')
print(f'  Reward acumulado: {baseline_total_reward:.2f}')
print(f'  Regret acumulado: {baseline_total_regret:.2f}')
print(f'  Regret medio: {np.mean(baseline_regrets):.4f}')

---
## Secao 5: Thompson Sampling (Algoritmo Adaptativo)

### Escolha do algoritmo: Thompson Sampling

**Justificativa:**
- Abordagem bayesiana que modela incerteza naturalmente
- Cada braco tem uma distribuicao Beta(alpha, beta) representando a crenca sobre sua taxa de conversao
- A exploracao acontece de forma probabilistica: bracos com mais incerteza tem mais chance de serem amostrados

### Estrategia de exploracao:
- **Prior:** Beta(1, 1) = Uniforme (sem conhecimento previo)
- **Atualizacao:** alpha += reward, beta += (1 - reward)
- **Decisao:** Amostrar theta ~ Beta(alpha, beta) para cada braco; escolher o maior

### Como o contexto entra na decisao:
- Segmentamos clientes por perfil (faixa etaria, saldo, historico)
- Cada segmento tem seus proprios parametros Beta por braco
- Isso permite personalizar a oferta por tipo de cliente

In [ ]:
# Implementacao do Thompson Sampling Contextual

class ThompsonSamplingContextual:
    """
    Thompson Sampling com segmentacao contextual.
    Cada segmento de cliente tem seus proprios priors Beta por braco.
    """
    
    def __init__(self, n_bracos, segmentos):
        self.n_bracos = n_bracos
        self.segmentos = segmentos
        # Prior Beta(1,1) para cada braco em cada segmento
        self.alpha = {seg: np.ones(n_bracos) for seg in segmentos}
        self.beta = {seg: np.ones(n_bracos) for seg in segmentos}
        self.historico = []
    
    def identificar_segmento(self, cliente):
        """Identifica o segmento do cliente baseado em suas features."""
        age = cliente.get('age', 40)
        balance = cliente.get('balance', 0)
        sucesso = cliente.get('sucesso_anterior', 0)
        
        if sucesso == 1:
            return 'historico_positivo'
        elif age < 30:
            return 'jovem'
        elif age > 55:
            return 'senior'
        elif balance > 1500:
            return 'alto_saldo'
        else:
            return 'padrao'
    
    def selecionar_braco(self, cliente):
        """Seleciona o braco via Thompson Sampling."""
        segmento = self.identificar_segmento(cliente)
        
        # Amostrar theta de cada braco
        thetas = np.array([
            np.random.beta(self.alpha[segmento][b], self.beta[segmento][b])
            for b in range(self.n_bracos)
        ])
        
        braco_escolhido = np.argmax(thetas)
        return braco_escolhido, segmento, thetas
    
    def atualizar(self, segmento, braco, reward):
        """Atualiza os priors com o reward observado."""
        self.alpha[segmento][braco] += reward
        self.beta[segmento][braco] += (1 - reward)
    
    def get_stats(self):
        """Retorna estatisticas dos priors por segmento."""
        stats = {}
        for seg in self.segmentos:
            media = self.alpha[seg] / (self.alpha[seg] + self.beta[seg])
            stats[seg] = {
                'alpha': self.alpha[seg].copy(),
                'beta': self.beta[seg].copy(),
                'media_estimada': media
            }
        return stats

# Definir segmentos
SEGMENTOS = ['jovem', 'senior', 'alto_saldo', 'historico_positivo', 'padrao']

print('Classe ThompsonSamplingContextual definida.')
print(f'Bracos: {len(BRACOS)}')
print(f'Segmentos: {SEGMENTOS}')

In [ ]:
# Simulacao do Thompson Sampling no conjunto de teste
ts = ThompsonSamplingContextual(n_bracos=len(BRACOS), segmentos=SEGMENTOS)

ts_rewards = []
ts_regrets = []
ts_bracos_escolhidos = []
ts_segmentos = []

for idx, row in df_test.iterrows():
    cliente = row.to_dict()
    
    # Selecionar braco via Thompson Sampling
    braco, segmento, thetas = ts.selecionar_braco(cliente)
    
    # Obter reward
    reward = simular_reward(row, braco)
    
    # Atualizar priors
    ts.atualizar(segmento, braco, reward)
    
    # Registrar
    ts_rewards.append(reward)
    ts_bracos_escolhidos.append(braco)
    ts_segmentos.append(segmento)
    
    # Regret
    best_reward = max(simular_reward(row, b) for b in BRACOS.keys())
    ts_regrets.append(best_reward - reward)

ts_ctr = np.mean(ts_rewards)
ts_total_reward = np.sum(ts_rewards)
ts_total_regret = np.sum(ts_regrets)

print('=' * 60)
print('RESULTADOS DO THOMPSON SAMPLING')
print('=' * 60)
print(f'  CTR (taxa de conversao): {ts_ctr:.4f}')
print(f'  Reward acumulado: {ts_total_reward:.2f}')
print(f'  Regret acumulado: {ts_total_regret:.2f}')
print(f'  Regret medio: {np.mean(ts_regrets):.4f}')
print(f'\nDistribuicao dos bracos escolhidos:')
for b in BRACOS.keys():
    count = ts_bracos_escolhidos.count(b)
    print(f'  Braco {b} ({BRACOS[b]}): {count} ({count/len(ts_bracos_escolhidos)*100:.1f}%)')

---
## Secao 6: Comparacao e Metricas

In [ ]:
# Comparacao Baseline vs Thompson Sampling
print('=' * 60)
print('COMPARACAO: BASELINE vs THOMPSON SAMPLING')
print('=' * 60)

comparacao = pd.DataFrame({
    'Metrica': ['CTR (Taxa de Conversao)', 'Reward Acumulado', 'Regret Acumulado', 'Regret Medio'],
    'Baseline': [f'{baseline_ctr:.4f}', f'{baseline_total_reward:.2f}', f'{baseline_total_regret:.2f}', f'{np.mean(baseline_regrets):.4f}'],
    'Thompson Sampling': [f'{ts_ctr:.4f}', f'{ts_total_reward:.2f}', f'{ts_total_regret:.2f}', f'{np.mean(ts_regrets):.4f}'],
    'Ganho TS vs Baseline': [
        f'{((ts_ctr - baseline_ctr) / baseline_ctr * 100):.1f}%' if baseline_ctr > 0 else 'N/A',
        f'{((ts_total_reward - baseline_total_reward) / baseline_total_reward * 100):.1f}%' if baseline_total_reward > 0 else 'N/A',
        f'{((baseline_total_regret - ts_total_regret) / baseline_total_regret * 100):.1f}%' if baseline_total_regret > 0 else 'N/A',
        f'{((np.mean(baseline_regrets) - np.mean(ts_regrets)) / np.mean(baseline_regrets) * 100):.1f}%' if np.mean(baseline_regrets) > 0 else 'N/A'
    ]
})

print(comparacao.to_string(index=False))

In [ ]:
# Grafico de Regret Acumulado ao longo do tempo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regret acumulado
axes[0].plot(np.cumsum(baseline_regrets), label='Baseline', color='#e74c3c', linewidth=2)
axes[0].plot(np.cumsum(ts_regrets), label='Thompson Sampling', color='#2ecc71', linewidth=2)
axes[0].set_title('Regret Acumulado ao Longo do Tempo')
axes[0].set_xlabel('Numero de decisoes')
axes[0].set_ylabel('Regret acumulado')
axes[0].legend()

# Reward acumulado
axes[1].plot(np.cumsum(baseline_rewards), label='Baseline', color='#e74c3c', linewidth=2)
axes[1].plot(np.cumsum(ts_rewards), label='Thompson Sampling', color='#2ecc71', linewidth=2)
axes[1].set_title('Reward Acumulado ao Longo do Tempo')
axes[1].set_xlabel('Numero de decisoes')
axes[1].set_ylabel('Reward acumulado')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Analise de exploracao: distribuicao dos bracos por segmento
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

df_analise = pd.DataFrame({'segmento': ts_segmentos, 'braco': ts_bracos_escolhidos})
pivot = df_analise.groupby(['segmento', 'braco']).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)

pivot_pct.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('Distribuicao de Bracos por Segmento (Thompson Sampling)')
ax.set_xlabel('Segmento')
ax.set_ylabel('Proporcao')
ax.legend(title='Braco', labels=[BRACOS[i] for i in range(len(BRACOS))])
plt.tight_layout()
plt.show()

In [ ]:
# Estatisticas finais dos priors (posterior)
print('=' * 60)
print('PRIORS FINAIS (POSTERIOR) POR SEGMENTO')
print('=' * 60)

stats = ts.get_stats()
for seg, data in stats.items():
    print(f'\nSegmento: {seg}')
    for b in range(len(BRACOS)):
        print(f'  Braco {b} ({BRACOS[b]}): '
              f'Beta(alpha={data["alpha"][b]:.1f}, beta={data["beta"][b]:.1f}) '
              f'-> media estimada = {data["media_estimada"][b]:.4f}')

---
## Secao 7: Golden Set (5 Casos de Teste)

Demonstracao com 5 perfis representativos de clientes, mostrando qual oferta o modelo recomendou e se a decisao faz sentido do ponto de vista de negocio.

In [ ]:
# Definir 5 perfis de clientes para o Golden Set
golden_set = [
    {
        'descricao': 'Jovem, estudante, baixo saldo',
        'age': 22, 'job': 'student', 'marital': 'single', 'education': 'tertiary',
        'default': 'no', 'balance': 200, 'housing': 'no', 'loan': 'no',
        'contact': 'cellular', 'campaign': 1, 'pdays': -1, 'previous': 0,
        'poutcome': 'unknown', 'sucesso_anterior': 0
    },
    {
        'descricao': 'Adulto, gerente, alto saldo',
        'age': 42, 'job': 'management', 'marital': 'married', 'education': 'tertiary',
        'default': 'no', 'balance': 5000, 'housing': 'yes', 'loan': 'no',
        'contact': 'cellular', 'campaign': 1, 'pdays': -1, 'previous': 0,
        'poutcome': 'unknown', 'sucesso_anterior': 0
    },
    {
        'descricao': 'Aposentado, saldo medio',
        'age': 62, 'job': 'retired', 'marital': 'married', 'education': 'secondary',
        'default': 'no', 'balance': 1200, 'housing': 'no', 'loan': 'no',
        'contact': 'telephone', 'campaign': 2, 'pdays': -1, 'previous': 0,
        'poutcome': 'unknown', 'sucesso_anterior': 0
    },
    {
        'descricao': 'Desempregado, sem contato anterior',
        'age': 38, 'job': 'unemployed', 'marital': 'divorced', 'education': 'primary',
        'default': 'no', 'balance': 100, 'housing': 'yes', 'loan': 'yes',
        'contact': 'telephone', 'campaign': 3, 'pdays': -1, 'previous': 0,
        'poutcome': 'unknown', 'sucesso_anterior': 0
    },
    {
        'descricao': 'Tecnico, contato anterior bem-sucedido',
        'age': 35, 'job': 'technician', 'marital': 'single', 'education': 'secondary',
        'default': 'no', 'balance': 800, 'housing': 'no', 'loan': 'no',
        'contact': 'cellular', 'campaign': 1, 'pdays': 100, 'previous': 2,
        'poutcome': 'success', 'sucesso_anterior': 1
    }
]

print(f'Golden Set definido com {len(golden_set)} perfis.')

In [ ]:
# Executar recomendacoes para o Golden Set
print('=' * 60)
print('GOLDEN SET: RECOMENDACOES')
print('=' * 60)

resultados_golden = []

for i, cliente in enumerate(golden_set, 1):
    braco, segmento, thetas = ts.selecionar_braco(cliente)
    prob_conversao = thetas[braco]
    
    resultado = {
        '#': i,
        'Perfil': cliente['descricao'],
        'Segmento': segmento,
        'Braco': braco,
        'Oferta': BRACOS[braco],
        'Prob. Conversao': f'{prob_conversao:.3f}'
    }
    resultados_golden.append(resultado)
    
    print(f'\nCliente {i}: {cliente["descricao"]}')
    print(f'  Segmento identificado: {segmento}')
    print(f'  Oferta recomendada: Braco {braco} - {BRACOS[braco]}')
    print(f'  Probabilidade de conversao: {prob_conversao:.3f}')

# Tabela resumo
print('\n' + '=' * 60)
print('TABELA RESUMO DO GOLDEN SET')
print('=' * 60)
df_golden = pd.DataFrame(resultados_golden)
print(df_golden.to_string(index=False))

### Analise de Coerencia do Golden Set

| # | Perfil | Oferta Esperada | Justificativa |
|---|--------|----------------|---------------|
| 1 | Jovem, estudante, baixo saldo | Produto alternativo | Jovens tendem a preferir emprestimos a depositos |
| 2 | Adulto, gerente, alto saldo | Deposito premium | Alto saldo indica capacidade e interesse em investir |
| 3 | Aposentado, saldo medio | Deposito padrao | Perfil conservador, busca seguranca |
| 4 | Desempregado, sem historico | Controle | Baixa propensao, evitar contato desnecessario |
| 5 | Tecnico, sucesso anterior | Deposito padrao/premium | Historico positivo indica alta propensao |

---
## Secao 8: Servico Demonstravel

Funcao Python que recebe dados de um cliente e retorna a oferta recomendada. Simula o comportamento de uma API REST sem necessidade de servidor.

In [ ]:
# Funcao de recomendacao (simula endpoint de API)

def recomendar_oferta(cliente: dict) -> dict:
    """
    Recebe dados de um cliente e retorna a oferta recomendada
    usando Thompson Sampling treinado.
    
    Parametros:
        cliente (dict): Dicionario com features do cliente
        
    Retorna:
        dict: Recomendacao com oferta, braco, probabilidade e confianca
    """
    braco, segmento, thetas = ts.selecionar_braco(cliente)
    prob = thetas[braco]
    
    # Definir nivel de confianca
    alpha_b = ts.alpha[segmento][braco]
    beta_b = ts.beta[segmento][braco]
    n_obs = alpha_b + beta_b - 2  # descontar prior
    
    if n_obs > 50:
        confianca = 'alta'
    elif n_obs > 20:
        confianca = 'media'
    else:
        confianca = 'baixa'
    
    return {
        'oferta_recomendada': BRACOS[braco],
        'braco_selecionado': braco,
        'segmento_cliente': segmento,
        'probabilidade_conversao': round(prob, 4),
        'confianca': confianca,
        'requer_revisao_humana': prob < 0.1
    }


def registrar_feedback(segmento: str, braco: int, converteu: bool):
    """
    Registra o feedback (conversao ou nao) para atualizar o modelo.
    Simula o endpoint POST /feedback de uma API.
    """
    reward = 1.0 if converteu else 0.0
    ts.atualizar(segmento, braco, reward)
    return {'status': 'ok', 'segmento': segmento, 'braco': braco, 'reward': reward}


print('Funcoes definidas:')
print('  - recomendar_oferta(cliente: dict) -> dict')
print('  - registrar_feedback(segmento, braco, converteu) -> dict')

In [ ]:
# Demonstracao: chamar a funcao para cada cliente do Golden Set
print('=' * 60)
print('DEMONSTRACAO DO SERVICO DE RECOMENDACAO')
print('=' * 60)

for i, cliente in enumerate(golden_set, 1):
    print(f'\n--- Cliente {i}: {cliente["descricao"]} ---')
    print(f'Input: age={cliente["age"]}, job={cliente["job"]}, balance={cliente["balance"]}')
    
    resultado = recomendar_oferta(cliente)
    
    print(f'Output:')
    for k, v in resultado.items():
        print(f'  {k}: {v}')
    
    # Simular feedback
    feedback = registrar_feedback(resultado['segmento_cliente'], resultado['braco_selecionado'], converteu=True)
    print(f'  Feedback registrado: {feedback["status"]}')

---
## Secao 9: MLflow Tracking

In [ ]:
# Configurar MLflow
mlflow.set_tracking_uri('file:///kaggle/working/mlruns')
mlflow.set_experiment('datathon-bandit')

print('MLflow configurado.')
print(f'Tracking URI: file:///kaggle/working/mlruns')
print(f'Experimento: datathon-bandit')

In [ ]:
# Registrar experimento do Baseline
with mlflow.start_run(run_name='baseline_regra_fixa'):
    mlflow.log_param('algoritmo', 'regra_fixa')
    mlflow.log_param('melhor_braco_fixo', melhor_braco_fixo)
    mlflow.log_param('n_bracos', len(BRACOS))
    mlflow.log_param('dataset', 'bank-marketing')
    
    mlflow.log_metric('ctr', baseline_ctr)
    mlflow.log_metric('reward_acumulado', baseline_total_reward)
    mlflow.log_metric('regret_acumulado', baseline_total_regret)
    mlflow.log_metric('regret_medio', np.mean(baseline_regrets))
    
    mlflow.set_tag('tipo', 'baseline')
    
    print('Experimento baseline registrado no MLflow.')
    print(f'  CTR: {baseline_ctr:.4f}')
    print(f'  Regret medio: {np.mean(baseline_regrets):.4f}')

In [ ]:
# Registrar experimento do Thompson Sampling
with mlflow.start_run(run_name='thompson_sampling_contextual'):
    mlflow.log_param('algoritmo', 'thompson_sampling')
    mlflow.log_param('contextual', True)
    mlflow.log_param('n_bracos', len(BRACOS))
    mlflow.log_param('n_segmentos', len(SEGMENTOS))
    mlflow.log_param('prior_alpha', 1)
    mlflow.log_param('prior_beta', 1)
    mlflow.log_param('dataset', 'bank-marketing')
    mlflow.log_param('n_amostras_teste', len(df_test))
    
    mlflow.log_metric('ctr', ts_ctr)
    mlflow.log_metric('reward_acumulado', ts_total_reward)
    mlflow.log_metric('regret_acumulado', ts_total_regret)
    mlflow.log_metric('regret_medio', np.mean(ts_regrets))
    mlflow.log_metric('ganho_vs_baseline_pct', ((ts_ctr - baseline_ctr) / baseline_ctr * 100) if baseline_ctr > 0 else 0)
    
    mlflow.set_tag('tipo', 'adaptativo')
    mlflow.set_tag('segmentos', str(SEGMENTOS))
    
    print('Experimento Thompson Sampling registrado no MLflow.')
    print(f'  CTR: {ts_ctr:.4f}')
    print(f'  Regret medio: {np.mean(ts_regrets):.4f}')
    print(f'  Ganho vs Baseline: {((ts_ctr - baseline_ctr) / baseline_ctr * 100):.1f}%' if baseline_ctr > 0 else '  Ganho vs Baseline: N/A')

In [ ]:
# Listar runs registrados
print('=' * 60)
print('EXPERIMENTOS REGISTRADOS NO MLFLOW')
print('=' * 60)

experiment = mlflow.get_experiment_by_name('datathon-bandit')
if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    print(runs[['run_id', 'tags.mlflow.runName', 'metrics.ctr', 'metrics.regret_medio']].to_string(index=False))
else:
    print('Experimento nao encontrado.')

---
## Secao 10: Conclusoes e Arquitetura Cloud

### Arquitetura-alvo em Nuvem (AWS)

Para colocar este projeto em producao, utilizariamos os seguintes servicos AWS:

**Amazon S3** para armazenamento dos dados brutos e processados, junto com logs de decisao do modelo. **Amazon ECS com Fargate** para hospedar a API de recomendacao em container Docker, exposta via **API Gateway** para receber requisicoes dos canais digitais. O modelo Thompson Sampling seria atualizado periodicamente via **AWS Step Functions** orquestrando um pipeline de retreino que le os feedbacks acumulados no S3.

Para monitoramento, **CloudWatch** coleta metricas de latencia, taxa de erro e drift do modelo. **SageMaker Feature Store** armazena as features dos clientes de forma centralizada, e os experimentos de ML sao rastreados via **MLflow hospedado em EC2** ou **SageMaker Experiments**.

```
Cliente -> API Gateway -> ECS (FastAPI) -> Modelo (Thompson Sampling)
                                                |
                                          S3 (logs de decisao)
                                                |
                                  Step Functions (retreino periodico)
```

### Limitacoes

- Dataset de uma unica instituicao portuguesa — pode nao generalizar para outros mercados
- Apenas 1 produto real no dataset; bracos sao simulados para demonstracao
- Dados estaticos, nao captura mudancas em tempo real (streaming)
- Desbalanceamento (~11% positivos) pode enviesar o bandit
- Sem dados de custo real por contato — usa reward binario como proxy
- Cold start: novos segmentos sem historico partem de prior uniforme
- Thompson Sampling assume independencia entre decisoes

### Governanca e Etica

- **Base legal:** Legitimo interesse para otimizacao de campanhas (LGPD Art. 10)
- **Finalidade:** Recomendar ofertas personalizadas maximizando conversao
- **Minimizacao:** Apenas features relevantes. Sem identificadores pessoais, genero ou raca
- **Retencao:** Dados de decisao retidos 12 meses para auditoria
- **Humano no loop:** Casos de baixa confianca sinalizados para revisao manual

### Conclusao

O Thompson Sampling contextual demonstrou capacidade de superar uma politica fixa (baseline), aprendendo progressivamente quais ofertas funcionam melhor para cada segmento de cliente. A abordagem equilibra exploracao e explotacao de forma natural, convergindo para melhores decisoes ao longo do tempo sem necessidade de testes A/B longos.

In [ ]:
print('=' * 60)
print('PROJETO CONCLUIDO COM SUCESSO!')
print('=' * 60)
print(f'\nResumo:')
print(f'  Dataset: bank-marketing ({df.shape[0]} registros)')
print(f'  Algoritmo: Thompson Sampling Contextual')
print(f'  Bracos: {len(BRACOS)} ofertas')
print(f'  Segmentos: {len(SEGMENTOS)}')
print(f'  CTR Baseline: {baseline_ctr:.4f}')
print(f'  CTR Thompson Sampling: {ts_ctr:.4f}')
print(f'  Ganho: {((ts_ctr - baseline_ctr) / baseline_ctr * 100):.1f}%' if baseline_ctr > 0 else '  Ganho: N/A')
print(f'\nGrupo 13 - Giovanna Catelli')